# M05-P02 — Embeddings & Semantic Search

En M05-P01 medimos cuánto contexto enviamos. Ahora cambiamos la pregunta:

```text
¿cómo encuentro solo la información relevante para una consulta?
```

Usaremos embeddings reales de Amazon Bedrock, similitud coseno y un pequeño corpus empresarial. Todavía no hay RAG ni base de datos vectorial: primero abrimos la caja del retrieval.

In [ ]:
import json
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA

from enterprise_genai.config import get_settings
from enterprise_genai.embeddings import (
    DEFAULT_DIMENSIONS,
    cosine_similarity,
    embed_documents,
    embed_text,
    semantic_search,
)

settings = get_settings()
CORPUS_PATH = Path('data/knowledge/m05_enterprise_chunks.json')
OUT_DIR = Path('outputs/m05_p02')
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('Modelo de embeddings:', settings.bedrock_embedding_model_id)
print('Región:', settings.aws_region)
print('Dimensiones usadas en la práctica:', DEFAULT_DIMENSIONS)

## 1. Un texto se convierte en un vector

Un embedding representa un texto como una lista de números. No intentes asignar significado humano a una coordenada individual: lo útil aparece al comparar vectores completos.

Titan Text Embeddings V2 puede devolver distintas dimensiones; en esta práctica usamos 512 y pedimos vectores normalizados.

In [ ]:
example = '73 usuarios no pueden autenticarse en la VPN.'
embedded = embed_text(example, dimensions=DEFAULT_DIMENSIONS, normalize=True)

print('Texto:', example)
print('Dimensiones:', embedded['dimensions'])
print('Norma del vector:', round(embedded['norm'], 6))
print('Tokens de entrada reportados por Titan:', embedded['input_tokens'])
print('Latencia:', embedded['latency_ms'], 'ms')
print('Primeras 12 coordenadas:')
print(np.array2string(embedded['embedding'][:12], precision=5))

## 2. Similitud semántica

Generamos varios embeddings y calculamos similitud coseno. Valores más altos indican que los vectores apuntan en direcciones más parecidas.

El score **no es una probabilidad** ni garantiza que dos textos digan exactamente lo mismo.

In [ ]:
sentences = [
    'No recibo el código MFA para entrar en la VPN.',
    'El segundo factor de autenticación no llega al teléfono.',
    'No puedo completar el acceso remoto porque falla la verificación en el móvil.',
    'El portal web corporativo responde lentamente.',
    'Hay que restaurar la base de datos desde un backup.',
    'Un usuario ha olvidado su contraseña.',
]

sentence_docs = [{'content': text} for text in sentences]
sentence_index = embed_documents(sentence_docs, dimensions=DEFAULT_DIMENSIONS)
sentence_matrix = sentence_index['matrix']

sim = np.zeros((len(sentences), len(sentences)))
for i in range(len(sentences)):
    for j in range(len(sentences)):
        sim[i, j] = cosine_similarity(sentence_matrix[i], sentence_matrix[j])

labels = [f'T{i+1}' for i in range(len(sentences))]
display(pd.DataFrame({'id': labels, 'texto': sentences}))
display(pd.DataFrame(sim, index=labels, columns=labels).round(3))

plt.figure(figsize=(7, 5))
plt.imshow(sim, vmin=-1, vmax=1)
plt.xticks(range(len(labels)), labels)
plt.yticks(range(len(labels)), labels)
plt.colorbar(label='Similitud coseno')
plt.title('Similitud entre embeddings')
plt.show()

## 3. Construimos un pequeño índice empresarial

El corpus ya está dividido en fragmentos pequeños para centrarnos en embeddings y retrieval. En M06 veremos chunking y RAG de forma más completa.

Cada fragmento conserva metadata: servicio, tipo de fuente, audiencia, título e identificador.

In [ ]:
documents = json.loads(CORPUS_PATH.read_text(encoding='utf-8'))
print('Fragmentos del corpus:', len(documents))
display(pd.DataFrame(documents)[['id', 'service', 'source_type', 'audience', 'title']])

corpus_index = embed_documents(documents, dimensions=DEFAULT_DIMENSIONS, normalize=True)
matrix = corpus_index['matrix']

index_df = pd.DataFrame({
    'id': [doc['id'] for doc in documents],
    'service': [doc['service'] for doc in documents],
    'source_type': [doc['source_type'] for doc in documents],
    'input_tokens': corpus_index['token_counts'],
    'embedding_latency_ms': corpus_index['latencies_ms'],
})

print('Forma de la matriz vectorial:', matrix.shape)
print('Llamadas al modelo:', corpus_index['request_count'])
display(index_df)

## 4. Buscar por significado

La consulta no necesita copiar literalmente el vocabulario del documento. La convertimos en embedding, calculamos similitud con todos los fragmentos y ordenamos de mayor a menor.

In [ ]:
QUERY = 'No me llega el código de seguridad al móvil cuando intento conectarme desde casa'
search_result = semantic_search(
    QUERY,
    documents,
    matrix,
    top_k=6,
    dimensions=DEFAULT_DIMENSIONS,
)

ranking_rows = []
for item in search_result['results']:
    doc = item['document']
    ranking_rows.append({
        'rank': item['rank'],
        'score': round(item['score'], 4),
        'id': doc['id'],
        'service': doc['service'],
        'source_type': doc['source_type'],
        'title': doc['title'],
    })

print('Consulta:', QUERY)
print('Tokens de la consulta:', search_result['query_tokens'])
print('Latencia embedding consulta:', search_result['query_latency_ms'], 'ms')
display(pd.DataFrame(ranking_rows))

## 5. Semántica no significa coincidencia literal

Para hacerlo visible, contamos cuántas palabras exactas comparte la consulta con cada resultado. Un fragmento puede ser semánticamente útil aunque use vocabulario distinto.

In [ ]:
def words(text):
    return set(re.findall(r'[a-záéíóúñü0-9]+', text.lower()))

query_words = words(QUERY)
overlap_rows = []
for item in search_result['results']:
    doc = item['document']
    shared = sorted(query_words & words(doc['content']))
    overlap_rows.append({
        'rank': item['rank'],
        'id': doc['id'],
        'cosine_score': round(item['score'], 4),
        'palabras_exactas_compartidas': len(shared),
        'ejemplos_overlap': ', '.join(shared[:8]),
    })

display(pd.DataFrame(overlap_rows))

## 6. `top-k` y filtros de metadata

`top-k` decide cuántos candidatos devolvemos. La metadata permite reducir el espacio de búsqueda usando información estructurada que no depende del embedding.

In [ ]:
print('Top-3 sin filtro:')
display(pd.DataFrame(ranking_rows[:3]))

filtered = semantic_search(
    QUERY,
    documents,
    matrix,
    top_k=5,
    filters={'service': 'VPN', 'source_type': 'runbook'},
    dimensions=DEFAULT_DIMENSIONS,
)

filtered_rows = []
for item in filtered['results']:
    doc = item['document']
    filtered_rows.append({
        'rank': item['rank'],
        'score': round(item['score'], 4),
        'id': doc['id'],
        'service': doc['service'],
        'source_type': doc['source_type'],
        'title': doc['title'],
    })

print('Filtro: service=VPN, source_type=runbook')
print('Candidatos después del filtro:', filtered['candidate_count'])
display(pd.DataFrame(filtered_rows))

## 7. Visualización 2D: útil, pero aproximada

Nuestros embeddings viven en 512 dimensiones. PCA los proyecta a 2 dimensiones para poder dibujarlos. La proyección pierde información: sirve para explorar, no para sustituir la similitud calculada en el espacio original.

In [ ]:
pca = PCA(n_components=2)
coords = pca.fit_transform(matrix)

plt.figure(figsize=(10, 6))
plt.scatter(coords[:, 0], coords[:, 1])
for i, doc in enumerate(documents):
    plt.annotate(doc['id'], (coords[i, 0], coords[i, 1]), fontsize=8, xytext=(4, 4), textcoords='offset points')
plt.xlabel('PCA 1')
plt.ylabel('PCA 2')
plt.title('Proyección 2D de embeddings del corpus')
plt.grid(alpha=0.2)
plt.show()

print('Varianza explicada por los 2 componentes:', round(float(pca.explained_variance_ratio_.sum()), 4))

## 8. Guardar el índice como artefacto

Todavía no usamos Chroma ni otra base vectorial. Guardamos la matriz y metadata para hacer tangible qué contiene un índice mínimo.

In [ ]:
np.savez_compressed(OUT_DIR / 'enterprise_embeddings.npz', embeddings=matrix)
metadata = {
    'model_id': settings.bedrock_embedding_model_id,
    'dimensions': DEFAULT_DIMENSIONS,
    'normalized': True,
    'documents': [
        {key: value for key, value in doc.items() if key != 'content'}
        for doc in documents
    ],
}
(OUT_DIR / 'enterprise_embeddings_metadata.json').write_text(
    json.dumps(metadata, ensure_ascii=False, indent=2),
    encoding='utf-8',
)

print('Vectores:', OUT_DIR / 'enterprise_embeddings.npz')
print('Metadata:', OUT_DIR / 'enterprise_embeddings_metadata.json')
print('Forma:', matrix.shape)

## 9. Evolución de `Knowledge`

Arranca la aplicación:

```bash
streamlit run app/Home.py
```

`Knowledge` permite consultar el mismo corpus, cambiar `top-k`, aplicar filtros e inspeccionar el embedding de consulta.

Todavía **no genera una respuesta con un LLM**. Ese límite es intencionado:

```text
M05-P02: query → embedding → retrieval
M06:     query → retrieval → contexto → LLM → respuesta con fuentes
```

## 10. MODIFICA

Prueba al menos una modificación:

- cambia la consulta por una sobre DNS, phishing o restauración de base de datos;
- compara `top-k=2` y `top-k=6`;
- filtra por `service` o `source_type`;
- cambia a 256 o 1024 dimensiones y compara tamaño y ranking;
- añade un nuevo fragmento al corpus y vuelve a construir el índice.

### Pregunta final

Si ya podemos recuperar fragmentos relevantes, ¿qué falta para convertir esto en un asistente que responda con conocimiento corporativo y cite sus fuentes?